# Official fast.ai course22 original 04 — Colab adaptation

Original: [04-how-does-a-neural-net-really-work.ipynb](https://github.com/fastai/course22/blob/230390584ba5b990e20487d68681a9ec5b258eff/clean/04-how-does-a-neural-net-really-work.ipynb)

Pinned source commit: `230390584ba5b990e20487d68681a9ec5b258eff`. Source SHA256: `0b7660cd2dbf70c01444b753efa781d30888a6c7627ba9158dc99192c791b063`.

This additional notebook preserves the full original computational experiments. Existing independently written companions 00–07 remain separate. Set QUICK_RUN=False; run every code cell. Source notebooks and authorship belong to Jeremy Howard and fast.ai. No separate course22 repository license was found at the pinned commit; no fastbook license is applied to these files.

- All original quadratic/MAE/gradient/ReLU experiments and widgets retained. y is explicitly a tensor for current PyTorch. Original gradient-accumulating loop is preserved and labelled; a separate10-step zero-grad comparison is added.

Public publication and competition submission have not been performed. Saved predictions are learning artifacts; the 418 Titanic test targets are unavailable. Original conceptual prompts/headings are retained below.


In [ ]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '03_fastai', 'fastai', '21_original_04_neural_net_foundations.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

In [ ]:
# Full-source execution and durable evidence helpers (Colab adaptation).
import hashlib, json, shutil, urllib.request
from datetime import datetime, timezone
if QUICK_RUN or os.environ.get('COURSE_SMOKE') == '1':
    raise ValueError('These original notebooks require full mode: set QUICK_RUN=False.')
RESULTS = COLAB_CONTEXT['folder'] / 'results' / Path(COLAB_CONTEXT['run']['notebook']).stem
MODELS = COLAB_CONTEXT['folder'] / 'models' / Path(COLAB_CONTEXT['run']['notebook']).stem
WORK = COLAB_CONTEXT['folder'] / 'data' / Path(COLAB_CONTEXT['run']['notebook']).stem
for directory in [RESULTS, MODELS, WORK]: directory.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

def record_stage(stage, **evidence):
    target = RESULTS / 'stage_evidence.json'
    started = COLAB_CONTEXT['run']['started_utc']
    data = json.loads(target.read_text(encoding='utf-8')) if target.is_file() else {}
    if data.get('run_started_utc') != started:
        data = {'notebook': COLAB_CONTEXT['run']['notebook'], 'run_started_utc': started, 'stages': {}}
    data['stages'][stage] = {'recorded_utc': datetime.now(timezone.utc).isoformat(), **evidence}
    data.update(quick_run=False, status='observed_stage_outputs', provider_course_completion_claimed=False,
                learner_personal_reflection_complete=False)
    temporary = target.with_suffix('.writing')
    temporary.write_text(json.dumps(data, indent=2, default=str), encoding='utf-8')
    temporary.replace(target)

def verified_download(url, target, sha256):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.is_file():
        temporary = target.with_name(target.name + '.download')
        with urllib.request.urlopen(url, timeout=120) as response, temporary.open('wb') as out:
            shutil.copyfileobj(response, out)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != sha256:
            raise ValueError('Source download checksum mismatch: ' + url)
        temporary.replace(target)
    if hashlib.sha256(target.read_bytes()).hexdigest() != sha256:
        raise ValueError('Existing source file differs from the pinned original: ' + str(target))
    return target

def save_plot(figure, name):
    if hasattr(figure, 'write_html'):
        figure.write_html(RESULTS / (name + '.html'), include_plotlyjs=True)
    elif hasattr(figure, 'savefig'):
        figure.savefig(RESULTS / (name + '.png'), dpi=150, bbox_inches='tight')
    return figure

record_stage('full_mode_setup', execution_location='hosted_colab' if IN_COLAB else 'local_validation',
             original_exercises_verified_complete=False)

## Evidence required for the original exercises

- Execute every nonempty original code cell top to bottom in hosted Colab; retain outputs and execution records.
- QUICK_RUN=False. Original experiment data coverage, epochs and learning rates are preserved.
- Original source attribution and the per-cell provenance map distinguish original exercises from companions.
- Save full notebook outputs and actual stage/checkpoint/artifact evidence in Drive.
- No provider badge, external publishing, competition submission or personal authorship is certified by execution.
- Retain noisy20-point quadratic MAE examples, widgets, explicit gradients, manual .01 update and10-step source loop, clipped/F.relu plots and double-ReLU exploration. Preserve the source accumulation demonstration and separately show gradient-reset behavior.


## Fitting a function with *gradient descent*

In [ ]:
from ipywidgets import interact
from fastai.basics import *

plt.rc('figure', dpi=90)

def plot_function(f, title=None, min=-2.1, max=2.1, color='r', ylim=None):
    x = torch.linspace(min,max, 100)[:,None]
    if ylim: plt.ylim(ylim)
    plt.plot(x, f(x), color)
    if title is not None: plt.title(title)

In [ ]:
def f(x): return 3*x**2 + 2*x + 1

plot_function(f, "$3x^2 + 2x + 1$")

In [ ]:
def quad(a, b, c, x): return a*x**2 + b*x + c

In [ ]:
def mk_quad(a,b,c): return partial(quad, a,b,c)

In [ ]:
f2 = mk_quad(3,2,1)
plot_function(f2)

In [ ]:
def noise(x, scale): return np.random.normal(scale=scale, size=x.shape)
def add_noise(x, mult, add): return x * (1+noise(x,mult)) + noise(x,add)

In [ ]:
np.random.seed(42)

x = torch.linspace(-2, 2, steps=20)[:,None]
y = add_noise(f(x), 0.15, 1.5)

y = torch.as_tensor(y, dtype=x.dtype)  # Current NumPy/PyTorch interoperation.

In [ ]:
x[:5],y[:5]

In [ ]:
plt.scatter(x,y);

In [ ]:
@interact(a=1.1, b=1.1, c=1.1)
def plot_quad(a, b, c):
    plt.scatter(x,y)
    plot_function(mk_quad(a,b,c), ylim=(-3,13))

In [ ]:
def mae(preds, acts): return (torch.abs(preds-acts)).mean()

In [ ]:
@interact(a=1.1, b=1.1, c=1.1)
def plot_quad(a, b, c):
    f = mk_quad(a,b,c)
    plt.scatter(x,y)
    loss = mae(f(x), y)
    plot_function(f, ylim=(-3,12), title=f"MAE: {loss:.2f}")

## Automating gradient descent

In [ ]:
def quad_mae(params):
    f = mk_quad(*params)
    return mae(f(x), y)

In [ ]:
quad_mae([1.1, 1.1, 1.1])

In [ ]:
abc = torch.tensor([1.1,1.1,1.1])

In [ ]:
abc.requires_grad_()

In [ ]:
loss = quad_mae(abc)
loss

In [ ]:
loss.backward()

In [ ]:
abc.grad

In [ ]:
with torch.no_grad():
    abc -= abc.grad*0.01
    loss = quad_mae(abc)
    
print(f'loss={loss:.2f}')

In [ ]:
for i in range(10):
    loss = quad_mae(abc)
    loss.backward()
    with torch.no_grad(): abc -= abc.grad*0.01
    print(f'step={i}; loss={loss:.2f}')

record_stage('source_mae_gradient_loop', steps=10, learning_rate=.01, coefficients=abc.detach().tolist(), loss=float(quad_mae(abc)), gradient_accumulation_preserved=True)

## How a neural network approximates any given function

In [ ]:
def rectified_linear(m,b,x):
    y = m*x+b
    return torch.clip(y, 0.)

In [ ]:
plot_function(partial(rectified_linear, 1,1))

In [ ]:
import torch.nn.functional as F
def rectified_linear2(m,b,x): return F.relu(m*x+b)
plot_function(partial(rectified_linear2, 1,1))

In [ ]:
@interact(m=1.5, b=1.5)
def plot_relu(m, b):
    plot_function(partial(rectified_linear, m,b), ylim=(-1,4))

In [ ]:
def double_relu(m1,b1,m2,b2,x):
    return rectified_linear(m1,b1,x) + rectified_linear(m2,b2,x)

@interact(m1=-1.5, b1=-1.5, m2=1.5, b2=1.5)
def plot_double_relu(m1, b1, m2, b2):
    plot_function(partial(double_relu, m1,b1,m2,b2), ylim=(-1,6))

# Additional worked comparison: independent per-step gradients require resetting .grad.
abc_independent = torch.tensor([1.1, 1.1, 1.1], requires_grad=True)
independent_history = []
for i in range(10):
    abc_independent.grad = None
    independent_loss = quad_mae(abc_independent)
    independent_loss.backward()
    with torch.no_grad(): abc_independent -= abc_independent.grad * .01
    independent_history.append(float(independent_loss))
record_stage('gradient_reset_comparison', steps=10, learning_rate=.01, loss_history=independent_history,
             coefficients=abc_independent.detach().tolist())
save_plot(plt.gcf(), 'double_relu_exploration')

## How to recognise an owl

In [ ]:
record_stage('reached_final_cell', all_previous_cells_independently_verified=False)
finish_run(COLAB_CONTEXT)